<a href="https://colab.research.google.com/github/akashde1998-Alpha/GCN-by-pytorch-/blob/main/GCN_via_pytorchgeometric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install Required Libraries
This cell installs `torch` and `torch_geometric`, which are essential libraries for building and working with Graph Neural Networks.

In [18]:
!pip install torch
!pip install torch_geometric



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.6 MB/s eta 0:00:00


### Import Necessary Modules
This cell imports various modules from `torch`, `torch.nn.functional`, and `torch_geometric` that will be used for defining the GCN model, handling data, and applying transformations.

In [19]:
import torch
import torch.nn.functional as F

import torch_geometric
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.datasets import Planetoid


### Load the Cora Dataset
This cell loads the Cora dataset, a commonly used benchmark for graph-based machine learning. `NormalizeFeatures()` is applied to normalize the node features, and the first graph in the dataset (`dataset[0]`) is assigned to the `data` variable.

In [20]:
dataset=torch_geometric.datasets.Planetoid(root='/Cora',name='Cora',transform=NormalizeFeatures())
data=dataset[0]

Processing...
Done!


### Inspect Dataset Properties
This cell prints various properties of the loaded Cora dataset, such as an example node's features, the total number of features per node, the number of classes, the total number of nodes, and the total number of edges. This helps in understanding the dataset's structure.

In [21]:
print(data)
print(data.x[36])
print(data.num_features)
print(dataset.num_classes)
print(data.num_nodes)
print(data.num_edges)


Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
tensor([0.0455, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000])
1433
7
2708
10556


### GCN Model Definition and Instantiation
This cell defines a `GCN` model with three convolutional layers. The first layer maps input features to 16 hidden channels, the second layer maps 16 channels to 10 channels, and the third layer maps to the number of output classes. It includes ReLU activation and dropout after the first two layers. An instance of this `GCN` model is then created with `hidden_channels=16` (though the intermediate channels are hardcoded within the class definition), and its architecture is printed.

In [22]:
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
  def __init__(self, hidden_channels):
    super().__init__()
    torch.manual_seed(123456)
    self.conv1=GCNConv(data.num_features,16)
    self.conv2=GCNConv(16, 10)
    self.conv3=GCNConv(10, dataset.num_classes)
  def forward(self, x, edge_index ):
     x=self.conv1(x, edge_index)
     x=x.relu()
     # Dropout: randomly drops features during training.
     x=F.dropout(x,p=0.5, training=self.training)   # This line can be removed if we do not want to use dropout.
     x=self.conv2(x, edge_index)
     x=x.relu()
     x=F.dropout(x,p=0.5, training=self.training)
     x=self.conv3(x, edge_index)
     return x

# Create an instance of the model and then print it
model = GCN(hidden_channels=16)
print(model)

GCN(
  (conv1): GCNConv(1433, 16)
  (conv2): GCNConv(16, 10)
  (conv3): GCNConv(10, 7)
)


### Optimizer and Loss Function Initialization
This cell initializes the `optimizer` (Stochastic Gradient Descent) which will be used to update the model's parameters during training, and the `criterion` (Cross Entropy Loss) which will measure the difference between the model's predictions and the true labels.

In [23]:
optimizer=torch.optim.SGD(model.parameters(), lr=0.01)
criterion=torch.nn.CrossEntropyLoss()

### Training Function Definition
This cell defines the `train` function, which performs a single training step. It sets the model to training mode, clears previous gradients, performs a forward pass, calculates the loss, computes gradients using backpropagation, and updates the model's parameters.

In [66]:
def train():
  model.train()
  optimizer.zero_grad()
  out=model(data.x, data.edge_index)
  loss=criterion(out[data.train_mask],data.y[data.train_mask])
  loss.backward()
  optimizer.step()
  return loss

### Test Function Definition

This cell defines the `test` function, which evaluates the model's performance on the test dataset. It sets the model to evaluation mode, performs a forward pass, calculates the predicted labels, and then computes the accuracy by comparing predictions with the true labels for the test set.

In [67]:
def test():
  model.eval()
  out=model(data.x, data.edge_index)
  pred=out.argmax(dim=1)
  test_correct=(pred[data.test_mask]==data.y[data.test_mask])
  test_acc=(int(test_correct.sum())/ int(data.test_mask.sum()))
  return test_acc


In [82]:
for epoch in range(1, 200):

    loss = train()

    print(f'Epoch: {epoch:03d}, ' f'Loss: {loss:.4f}')
    #print(f"Epoch: {epoch}, Loss: {loss}")
    #print("Epoch:", epoch, "Loss:", loss)

Epoch: 001, Loss: 1.9433
Epoch: 002, Loss: 1.9414
Epoch: 003, Loss: 1.9428
Epoch: 004, Loss: 1.9425
Epoch: 005, Loss: 1.9416
Epoch: 006, Loss: 1.9439
Epoch: 007, Loss: 1.9434
Epoch: 008, Loss: 1.9430
Epoch: 009, Loss: 1.9431
Epoch: 010, Loss: 1.9434
Epoch: 011, Loss: 1.9423
Epoch: 012, Loss: 1.9428
Epoch: 013, Loss: 1.9416
Epoch: 014, Loss: 1.9431
Epoch: 015, Loss: 1.9419
Epoch: 016, Loss: 1.9434
Epoch: 017, Loss: 1.9423
Epoch: 018, Loss: 1.9427
Epoch: 019, Loss: 1.9433
Epoch: 020, Loss: 1.9428
Epoch: 021, Loss: 1.9416
Epoch: 022, Loss: 1.9429
Epoch: 023, Loss: 1.9432
Epoch: 024, Loss: 1.9421
Epoch: 025, Loss: 1.9412
Epoch: 026, Loss: 1.9433
Epoch: 027, Loss: 1.9444
Epoch: 028, Loss: 1.9438
Epoch: 029, Loss: 1.9432
Epoch: 030, Loss: 1.9420
Epoch: 031, Loss: 1.9442
Epoch: 032, Loss: 1.9440
Epoch: 033, Loss: 1.9430
Epoch: 034, Loss: 1.9436
Epoch: 035, Loss: 1.9423
Epoch: 036, Loss: 1.9432
Epoch: 037, Loss: 1.9439
Epoch: 038, Loss: 1.9446
Epoch: 039, Loss: 1.9430
Epoch: 040, Loss: 1.9435


### Evaluate Model Performance

This cell evaluates the trained model on the test dataset and prints the final test accuracy. The `test()` function calculates how well the model predicts the correct labels for the nodes in the test set.

In [95]:
test_acc = test()
print("test_acc:", test_acc)

test_acc: 0.164
